# 线缆布线优化算法 - 使用示例

**作者**: 智子 (Sophon)  
**创建日期**: 2026-03-18  
**版本**: 1.0.0

本 Notebook 展示了线缆布线优化算法库的完整使用示例，涵盖从基础到高级的 8 个场景。

## 目录

1. [环境准备](#1-环境准备)
2. [示例 1: Dijkstra 最短路径](#2-示例 1-dijkstra-最短路径)
3. [示例 2: 遗传算法优化](#3-示例 2-遗传算法优化)
4. [示例 3: VNS 局部优化](#4-示例 3-vns-局部优化)
5. [示例 4: NSGA-II 多目标优化](#5-示例 4-nsga-ii 多目标优化)
6. [示例 5: 大规模问题分解](#6-示例 5-大规模问题分解)
7. [示例 6: 智能算法选择器](#7-示例 6-智能算法选择器)
8. [示例 7: 算法性能对比](#8-示例 7-算法性能对比)
9. [示例 8: 强化学习路径规划](#9-示例 8-强化学习路径规划)

---

## 1. 环境准备

In [ ]:
# 导入必要的库
import sys
import os

# 添加项目路径
sys.path.insert(0, os.path.abspath('..'))

# 通用导入
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
from typing import List, Tuple, Dict
import time

# 设置中文字体
plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

print("✅ 环境准备完成")

---
## 2. 示例 1: Dijkstra 最短路径

**场景**: 在已知网络拓扑中找到两点之间的最短路径  
**适用**: 精确求解、小规模问题、需要最优解的场景

In [ ]:
# 导入 Dijkstra 算法
from examples.02_dijkstra import DijkstraSolver, visualize_path

# 创建测试问题
np.random.seed(42)
n_nodes = 20
coordinates = np.random.rand(n_nodes, 2) * 100

# 构建距离矩阵
dist_matrix = np.zeros((n_nodes, n_nodes))
for i in range(n_nodes):
    for j in range(n_nodes):
        dist_matrix[i, j] = np.sqrt(np.sum((coordinates[i] - coordinates[j])**2))

# 求解最短路径 (从节点 0 到节点 19)
solver = DijkstraSolver(dist_matrix)
path, cost = solver.solve(start=0, end=n_nodes-1)

print(f"📍 节点数：{n_nodes}")
print(f"🛤️  最短路径：{path}")
print(f"📏 路径长度：{cost:.2f}")
print(f"⏱️  计算时间：{solver.solve_time:.4f}秒")

In [ ]:
# 可视化结果
fig, ax = plt.subplots(figsize=(10, 8))

# 绘制所有节点
ax.scatter(coordinates[:, 0], coordinates[:, 1], c='lightblue', s=100, edgecolors='blue', linewidths=2)

# 绘制最短路径
path_coords = coordinates[path]
ax.plot(path_coords[:, 0], path_coords[:, 1], 'r-', linewidth=2, marker='o', markersize=8)

# 标记起点和终点
ax.scatter(coordinates[0, 0], coordinates[0, 1], c='green', s=200, marker='s', label='起点', zorder=5)
ax.scatter(coordinates[-1, 0], coordinates[-1, 1], c='red', s=200, marker='*', label='终点', zorder=5)

ax.set_xlabel('X 坐标')
ax.set_ylabel('Y 坐标')
ax.set_title('Dijkstra 最短路径示例')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 3. 示例 2: 遗传算法优化

**场景**: TSP 类回路优化问题  
**适用**: 中等规模问题、全局搜索、并行计算

In [ ]:
# 导入遗传算法
from examples.03_genetic_algorithm import GeneticAlgorithm, CableRoutingGA

# 创建测试问题
np.random.seed(42)
n_nodes = 30
coordinates = np.random.rand(n_nodes, 2) * 100

# 构建距离矩阵
dist_matrix = np.zeros((n_nodes, n_nodes))
for i in range(n_nodes):
    for j in range(n_nodes):
        dist_matrix[i, j] = np.sqrt(np.sum((coordinates[i] - coordinates[j])**2))

# 配置 GA 参数
ga = GeneticAlgorithm(
    n_nodes=n_nodes,
    dist_matrix=dist_matrix,
    population_size=100,
    crossover_rate=0.85,
    mutation_rate=0.15,
    generations=200,
    elite_ratio=0.1,
    random_seed=42
)

# 运行优化
best_path, best_cost, history = ga.optimize(verbose=True)

print(f"\n📊 优化结果:")
print(f"📍 节点数：{n_nodes}")
print(f"🛤️  最优路径成本：{best_cost:.2f}")
print(f"⏱️  计算时间：{ga.execution_time:.2f}秒")
print(f"📈 收敛代数：{ga.convergence_generation}")

In [ ]:
# 可视化收敛曲线
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# 收敛曲线
ax1.plot(history['best_fitness'], 'b-', linewidth=2, label='最优适应度')
ax1.plot(history['avg_fitness'], 'g--', linewidth=2, label='平均适应度')
ax1.set_xlabel('代数')
ax2.set_ylabel('路径成本')
ax1.set_title('遗传算法收敛曲线')
ax1.legend()
ax1.grid(True, alpha=0.3)

# 路径可视化
path_coords = coordinates[best_path]
ax2.plot(path_coords[:, 0], path_coords[:, 1], 'bo-', linewidth=2, markersize=6)
ax2.plot([path_coords[-1, 0], path_coords[0, 0]], 
         [path_coords[-1, 1], path_coords[0, 1]], 'bo-', linewidth=2)  # 闭合回路
ax2.set_xlabel('X 坐标')
ax2.set_ylabel('Y 坐标')
ax2.set_title(f'GA 最优路径 (成本={best_cost:.2f})')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 4. 示例 3: VNS 局部优化

**场景**: 在初始解基础上进行局部优化  
**适用**: 中等规模问题、需要高质量近似解

In [ ]:
# 导入 VNS 算法
from examples.08_variable_neighborhood_search import VariableNeighborhoodSearch, CableRoutingVNS

# 创建测试问题
np.random.seed(42)
n_nodes = 30
coordinates = np.random.rand(n_nodes, 2) * 100

# 构建距离矩阵
dist_matrix = np.zeros((n_nodes, n_nodes))
for i in range(n_nodes):
    for j in range(n_nodes):
        dist_matrix[i, j] = np.sqrt(np.sum((coordinates[i] - coordinates[j])**2))

# 创建 VNS 求解器
vns = CableRoutingVNS(
    coordinates=coordinates,
    k_max=5,
    max_iterations=500,
    max_no_improve=100,
    random_seed=42
)

# 生成初始解 (最近邻)
initial_path, initial_cost = vns.generate_initial_solution(method='nearest')
print(f"📊 初始解 (最近邻): 成本 = {initial_cost:.2f}")

# 运行 VNS 优化
best_path, best_cost, history = vns.optimize(verbose=True)

improvement = (initial_cost - best_cost) / initial_cost * 100
print(f"\n📊 VNS 优化结果:")
print(f"🎯 优化后成本：{best_cost:.2f}")
print(f"📈 改进幅度：{improvement:.2f}%")
print(f"⏱️  计算时间：{vns.execution_time:.2f}秒")
print(f"🔄 迭代次数：{history['iterations']}")

In [ ]:
# 可视化 VNS 优化过程
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# 收敛曲线
ax1.plot(history['best_costs'], 'r-', linewidth=2, label='VNS 最优成本')
ax1.axhline(y=initial_cost, color='gray', linestyle='--', label=f'初始解 ({initial_cost:.2f})')
ax1.set_xlabel('迭代次数')
ax1.set_ylabel('路径成本')
ax1.set_title('VNS 收敛曲线')
ax1.legend()
ax1.grid(True, alpha=0.3)

# 路径对比
initial_coords = coordinates[initial_path]
best_coords = coordinates[best_path]

ax2.plot(initial_coords[:, 0], initial_coords[:, 1], 'gray', alpha=0.5, linewidth=1, label='初始解')
ax2.plot(best_coords[:, 0], best_coords[:, 1], 'r-', linewidth=2, marker='o', markersize=5, label='VNS 优化后')
ax2.scatter(coordinates[:, 0], coordinates[:, 1], c='blue', s=50, alpha=0.5)
ax2.set_xlabel('X 坐标')
ax2.set_ylabel('Y 坐标')
ax2.set_title('路径对比')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 5. 示例 4: NSGA-II 多目标优化

**场景**: 同时优化多个冲突目标 (如成本 vs 风险)  
**适用**: 需要权衡多个目标的决策场景

In [ ]:
# 导入 NSGA-II 算法
from examples.18_multiobjective_nsga2 import NSGA2Optimizer, MultiObjectiveCableRouting

# 创建测试问题 (带风险)
np.random.seed(42)
n_nodes = 20
coordinates = np.random.rand(n_nodes, 2) * 100
risk_values = np.random.rand(n_nodes)  # 每个节点的风险值

# 构建距离矩阵和风险矩阵
dist_matrix = np.zeros((n_nodes, n_nodes))
risk_matrix = np.zeros((n_nodes, n_nodes))
for i in range(n_nodes):
    for j in range(n_nodes):
        dist_matrix[i, j] = np.sqrt(np.sum((coordinates[i] - coordinates[j])**2))
        risk_matrix[i, j] = (risk_values[i] + risk_values[j]) / 2  # 边风险为两端点风险平均

# 创建 NSGA-II 求解器
nsga2 = NSGA2Optimizer(
    n_nodes=n_nodes,
    dist_matrix=dist_matrix,
    risk_matrix=risk_matrix,
    population_size=100,
    n_generations=200,
    crossover_rate=0.9,
    mutation_rate=0.15,
    random_seed=42
)

# 运行优化
pareto_front, pareto_paths = nsga2.optimize(verbose=True)

print(f"\n📊 NSGA-II 优化结果:")
print(f"📍 节点数：{n_nodes}")
print(f"🎯 Pareto 前沿大小：{len(pareto_front)}")
print(f"⏱️  计算时间：{nsga2.execution_time:.2f}秒")

# 展示极端解和折中解
if len(pareto_front) > 0:
    min_length_idx = np.argmin([sol[0] for sol in pareto_front])
    min_risk_idx = np.argmin([sol[1] for sol in pareto_front])
    
    # 折中解：最接近理想点
    ideal_point = np.array([min([s[0] for s in pareto_front]), min([s[1] for s in pareto_front])])
    distances_to_ideal = [np.sqrt((s[0]-ideal_point[0])**2 + (s[1]-ideal_point[1])**2) for s in pareto_front]
    compromise_idx = np.argmin(distances_to_ideal)
    
    print(f"\n🔹 最小长度解：长度={pareto_front[min_length_idx][0]:.2f}, 风险={pareto_front[min_length_idx][1]:.2f}")
    print(f"🔹 最小风险解：长度={pareto_front[min_risk_idx][0]:.2f}, 风险={pareto_front[min_risk_idx][1]:.2f}")
    print(f"🔹 折中解：长度={pareto_front[compromise_idx][0]:.2f}, 风险={pareto_front[compromise_idx][1]:.2f}")

In [ ]:
# 可视化 Pareto 前沿
fig, ax = plt.subplots(figsize=(10, 8))

# 绘制 Pareto 前沿
front_array = np.array(pareto_front)
ax.scatter(front_array[:, 0], front_array[:, 1], c='red', s=100, marker='o', label='Pareto 最优解', zorder=5)

# 连接 Pareto 前沿点
sorted_idx = np.argsort(front_array[:, 0])
ax.plot(front_array[sorted_idx, 0], front_array[sorted_idx, 1], 'r--', linewidth=2, alpha=0.5)

ax.set_xlabel('路径长度')
ax.set_ylabel('路径风险')
ax.set_title('NSGA-II Pareto 前沿 (长度 vs 风险)')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 6. 示例 5: 大规模问题分解

**场景**: 100+ 节点的大规模布线问题  
**适用**: 大规模问题、需要快速获得可接受解

In [ ]:
# 导入分解算法
from examples.19_large_scale_decomposition import DecompositionOptimizer

# 创建大规模测试问题
np.random.seed(42)
n_nodes = 100
coordinates = np.random.rand(n_nodes, 2) * 100

# 创建分解优化器
decomposer = DecompositionOptimizer(
    coordinates=coordinates,
    n_clusters=10,  # 分为 10 个簇
    internal_iterations=100,
    global_iterations=50,
    random_seed=42
)

# 运行优化
full_path, total_cost, details = decomposer.optimize(verbose=True)

print(f"\n📊 分解算法结果:")
print(f"📍 节点数：{n_nodes}")
print(f"🔷 簇数量：{details['n_clusters']}")
print(f"📏 总路径长度：{total_cost:.2f}")
print(f"⏱️  计算时间：{details['total_time']:.2f}秒")
print(f"📊 簇间路径长度：{details['inter_cluster_cost']:.2f}")
print(f"📊 簇内路径总长：{details['intra_cluster_cost']:.2f}")

In [ ]:
# 可视化分解结果
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# 左图：簇划分
scatter = ax1.scatter(coordinates[:, 0], coordinates[:, 1], 
                      c=details['cluster_labels'], cmap='tab10', s=100, edgecolors='black')
ax1.set_xlabel('X 坐标')
ax1.set_ylabel('Y 坐标')
ax1.set_title(f'节点聚类 (k={details["n_clusters"]})')
plt.colorbar(scatter, ax=ax1, label='簇编号')
ax1.grid(True, alpha=0.3)

# 右图：最终路径
path_coords = coordinates[full_path]
ax2.plot(path_coords[:, 0], path_coords[:, 1], 'b-', linewidth=1.5, marker='o', markersize=3)
ax2.scatter(coordinates[:, 0], coordinates[:, 1], c=details['cluster_labels'], 
            cmap='tab10', s=50, alpha=0.5)
ax2.set_xlabel('X 坐标')
ax2.set_ylabel('Y 坐标')
ax2.set_title(f'优化后路径 (总长度={total_cost:.2f})')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 7. 示例 6: 智能算法选择器

**场景**: 根据问题特征自动推荐最佳算法  
**适用**: 快速决策、算法选择困难时

In [ ]:
# 导入算法选择器
from examples.21_algorithm_selector import AlgorithmSelector, ProblemCharacteristics

# 定义问题特征
problem = ProblemCharacteristics(
    n_nodes=50,
    requires_exact_solution=False,
    time_limit_seconds=10,
    has_dynamic_obstacles=False,
    has_multiple_objectives=False,
    has_capacity_constraints=False,
    graph_density='medium',
    solution_quality_priority='balanced'
)

# 创建选择器并获取推荐
selector = AlgorithmSelector()
recommendations = selector.recommend(problem, top_k=3)

print("📋 问题特征:")
print(f"  • 节点数：{problem.n_nodes}")
print(f"  • 需要精确解：{problem.requires_exact_solution}")
print(f"  • 时间限制：{problem.time_limit_seconds}秒")
print(f"  • 多目标：{problem.has_multiple_objectives}")
print(f"  • 质量优先级：{problem.solution_quality_priority}")

print(f"\n🎯 推荐算法 (Top 3):")
for i, rec in enumerate(recommendations, 1):
    print(f"\n{i}. {rec['algorithm']} (得分：{rec['score']:.2f})")
    print(f"   理由：{rec['reason']}")
    print(f"   预期性能：{rec['expected_performance']}")

In [ ]:
# 可视化算法评分
fig, ax = plt.subplots(figsize=(12, 8))
selector.plot_comparison(problem, ax=ax)
plt.tight_layout()
plt.show()

---
## 8. 示例 7: 算法性能对比

**场景**: 在统一测试集上对比多种算法性能  
**适用**: 算法选型、性能基准测试

In [ ]:
# 导入对比框架
from examples.11_algorithm_comparison import AlgorithmBenchmark

# 创建测试问题
np.random.seed(42)
n_nodes = 20
coordinates = np.random.rand(n_nodes, 2) * 100

# 构建距离矩阵
dist_matrix = np.zeros((n_nodes, n_nodes))
for i in range(n_nodes):
    for j in range(n_nodes):
        dist_matrix[i, j] = np.sqrt(np.sum((coordinates[i] - coordinates[j])**2))

# 创建基准测试器
benchmark = AlgorithmBenchmark(dist_matrix, random_seed=42)

# 运行对比实验
results = benchmark.run_comparison(algorithms=['dijkstra', 'ga', 'vns', 'ts', 'aco'], verbose=True)

# 打印结果表格
print("\n" + "="*70)
print(f"{'算法':<20} {'成本':<12} {'时间 (s)':<12} {'排名':<8}")
print("="*70)

sorted_results = sorted(results.items(), key=lambda x: x[1]['cost'])
for rank, (algo, res) in enumerate(sorted_results, 1):
    print(f"{algo:<20} {res['cost']:<12.2f} {res['time']:<12.4f} #{rank:<8}")

print("="*70)

In [ ]:
# 可视化对比结果
benchmark.plot_comparison(results)

---
## 9. 示例 8: 强化学习路径规划

**场景**: 网格环境中的动态路径规划  
**适用**: 动态环境、需要适应性的场景

In [ ]:
# 导入 Q-Learning
from examples.12_dqn_reinforcement_learning import CableRoutingEnv, QLearningTrainer

# 创建环境
env = CableRoutingEnv(
    grid_size=10,
    n_obstacles=15,
    start=(0, 0),
    goal=(9, 9),
    random_seed=42
)

# 创建训练器
trainer = QLearningTrainer(
    env=env,
    learning_rate=0.1,
    discount_factor=0.95,
    epsilon_start=1.0,
    epsilon_end=0.01,
    epsilon_decay=0.995,
    random_seed=42
)

# 训练
print("🚀 开始训练 Q-Learning 智能体...")
history = trainer.train(n_episodes=300, verbose=True)

print(f"\n📊 训练完成:")
print(f"  • 训练回合：300")
print(f"  • 最终平均奖励：{history['avg_rewards'][-1]:.2f}")
print(f"  • 最佳奖励：{max(history['best_rewards']):.2f}")

In [ ]:
# 可视化训练结果
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. 训练曲线
axes[0].plot(history['avg_rewards'], 'b-', linewidth=2, label='平均奖励')
axes[0].plot(history['best_rewards'], 'r--', linewidth=2, label='最佳奖励')
axes[0].set_xlabel('回合')
axes[0].set_ylabel('奖励')
axes[0].set_title('训练曲线')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# 2. Q 表热力图
trainer.plot_q_table(ax=axes[1])
axes[1].set_title('Q 表策略热力图')

# 3. 测试路径
trainer.plot_test_path(ax=axes[2])
axes[2].set_title('测试路径')

plt.tight_layout()
plt.show()

---
## 总结

本 Notebook 展示了线缆布线优化算法库的 8 个核心使用场景:

| 示例 | 算法 | 适用场景 | 规模 |
--------|------|----------|------|
1 | Dijkstra | 精确最短路径 | 任意 |
2 | 遗传算法 | TSP 回路优化 | 中等 (30-100) |
3 | VNS | 局部优化 | 中等 (20-80) |
4 | NSGA-II | 多目标优化 | 中小 (15-50) |
5 | 分解算法 | 大规模问题 | 大 (100-500+) |
6 | 算法选择器 | 智能推荐 | 任意 |
7 | 算法对比 | 性能基准 | 任意 |
8 | Q-Learning | 动态路径规划 | 中小网格 |

### 下一步

- 查看 `docs/algorithm-notes.md` 了解算法理论
- 查看 `docs/usage-examples.md` 了解更多使用示例
- 查看 `README.md` 了解项目完整说明

---

**Happy Optimizing! 🚀**